In [1]:
import time # for delays


# NOTE: This is the 'business_logic_3' notebook:
It handles Goal 3: The Instant Translator Hotkey. When the user selects a word and presses ctral+c after that he presses Ctrl+Shift+T, it copies the word and past it in Google Translate automatically.

In [3]:
KEYBOARD_AVAILABLE = False # set default flag
WIN32_AVAILABLE = False # set default flag

try:
    import keyboard # import keyboard library
    KEYBOARD_AVAILABLE = True # set flag to true
except ImportError:
    print("WARNING: Install keyboard with: !pip install keyboard") # print warning

try:
    import win32gui # import win32gui
    import win32con # import win32con
    import win32clipboard # import win32clipboard
    WIN32_AVAILABLE = True # set flag to true
except ImportError:
    print("WARNING: Install pywin32 with: !pip install pywin32") # print warning

class BusinessLogic3: # define the main class for goal 3
    def __init__(self): # define constructor
        self.is_listening = False # set hotkey state
        self.last_translated = "" # set translated word
        self.current_hotkey = 'win+ctrl+z' # set default hotkey
        
    def find_chrome_window(self): # define method to find chrome window
        if not WIN32_AVAILABLE: # check if available
            return None # return none
        chrome_windows = [] # create list for chrome windows
        
        def callback(hwnd, results): # define callback for each window
            if win32gui.IsWindowVisible(hwnd): # check if visible
                title = win32gui.GetWindowText(hwnd).lower() # get title
                if "chrome" in title: # check if chrome
                    results.append(hwnd) # add to list
        
        win32gui.EnumWindows(callback, chrome_windows) # search all windows
        return chrome_windows[0] if chrome_windows else None # return first chrome

    def get_clipboard_text(self): # define method to read text from clipboard
        if not WIN32_AVAILABLE: # check if available
            return "" # return empty
        try:
            win32clipboard.OpenClipboard() # open clipboard
            try:
                data = win32clipboard.GetClipboardData(win32clipboard.CF_UNICODETEXT) # read text
            except TypeError:
                data = "" # set empty if no text
            finally:
                win32clipboard.CloseClipboard() # close clipboard
            return data.strip() # return the text
        except Exception:
            return "" # return empty on error

    def paste_to_open_translate(self): # define method to paste to google translate
        if not WIN32_AVAILABLE: # check if available
            return False, "pywin32 not installed." # return error
        
        text = self.get_clipboard_text() # read clipboard directly
        
        if not text: # check if clipboard has text
            return False, "Clipboard empty. Select text and press Ctrl+C first." # return error
        
        chrome_hwnd = self.find_chrome_window() # find chrome window
        if not chrome_hwnd: # check if found
            return False, "Chrome not found. Open Chrome with Google Translate." # return error
        
        original_window = win32gui.GetForegroundWindow() # save current window
        
        try:
            win32gui.ShowWindow(chrome_hwnd, win32con.SW_RESTORE) # restore if minimized
            win32gui.SetForegroundWindow(chrome_hwnd) # bring to front
        except Exception:
            pass # ignore error
        
        time.sleep(0.5) # wait for chrome to focus
        
        keyboard.send('ctrl+a') # select all in input
        time.sleep(0.1) # small delay
        keyboard.send('ctrl+v') # paste the word
        time.sleep(0.3) # wait for paste
        
        try:
            win32gui.SetForegroundWindow(original_window) # switch back to original window
        except Exception:
            pass # ignore error
        
        self.last_translated = text # save last word
        return True, f"'{text}' pasted to Google Translate!" # return success

    def on_hotkey_pressed(self): # define method run when win+ctrl+z is pressed
        self.paste_to_open_translate() # call paste method
            
    def start_listener(self, custom_hotkey='win+ctrl+z'): # define method to start the hotkey listener
        if not KEYBOARD_AVAILABLE: # check if keyboard available
            return False, "keyboard not installed. Run: !pip install keyboard" # return error
        if not self.is_listening: # check if not running
            keyboard.add_hotkey(self.current_hotkey, self.on_hotkey_pressed) # add hotkey
            self.is_listening = True # set flag
            return True, f"ACTIVE! Hotkey set to: [{self.current_hotkey}]" # return success
        return False, "Already running." # return already running

    def stop_listener(self): # define method to stop the hotkey listener
        if not KEYBOARD_AVAILABLE: # check if available
            return False, "keyboard not installed." # return error
        if not self.is_listening: # check if running
            return False, "Not running." # return not running
        try:
            keyboard.remove_hotkey(self.current_hotkey) # remove hotkey
        except Exception:
            pass # ignore error
        self.is_listening = False # set flag
        return True, "DEACTIVATED." # return success

print("SUCCESS: BusinessLogic3 loaded! Use Ctrl+C to copy, then press win+ctrl+z to translate.") # print successfully

SUCCESS: BusinessLogic3 loaded! Use Ctrl+C to copy, then press win+ctrl+z to translate.
